# RSNA 2024 — Download Study 100206310 (3 Series)

**Purpose:** Extract all three MRI series for study `100206310` from the full RSNA competition dataset and package them into a small, downloadable ZIP (~50–200 MB) instead of downloading the entire 35 GB dataset.

## What this notebook produces

```
rsna_study_100206310.zip
├── train_images/
│   └── 100206310/
│       ├── 1012284084/   ← Axial T2          (all slices)
│       ├── 1792451510/   ← Sagittal T2/STIR  (all slices)
│       └── 2092806862/   ← Sagittal T1       (all slices)
└── metadata/
    ├── train_series_descriptions.csv   ← filtered to study 100206310
    ├── train_label_coordinates.csv     ← filtered to study 100206310
    └── train.csv                       ← row for study 100206310
```

## Instructions
1. On Kaggle: **File → Add input** → add the competition dataset `rsna-2024-lumbar-spine-degenerative-classification`
2. Run all cells
3. Go to the **Output** tab → download `rsna_study_100206310.zip`
4. Extract locally into `./data/rsna_study_100206310/` and run `rsna_lumbar_spine_medgemma.ipynb`

## Step 1 — Verify the competition data is mounted

In [ ]:
import os
from pathlib import Path

COMPETITION_ROOT = Path("/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification")
TRAIN_IMAGES     = COMPETITION_ROOT / "train_images"
STUDY_ID         = "100206310"

# The three series for study 100206310 (from train_series_descriptions.csv)
SERIES = {
    "1012284084": "Axial T2",
    "1792451510": "Sagittal T2/STIR",
    "2092806862": "Sagittal T1",
}

print(f"Competition root  : {COMPETITION_ROOT}")
print(f"Exists            : {COMPETITION_ROOT.exists()}")
print()

study_path = TRAIN_IMAGES / STUDY_ID
print(f"Study path        : {study_path}")
print(f"Study exists      : {study_path.exists()}")
print()

total_files = 0
for series_id, description in SERIES.items():
    series_path = study_path / series_id
    n_dcm = len(list(series_path.glob("*.dcm"))) if series_path.exists() else 0
    total_files += n_dcm
    status = "✓" if series_path.exists() else "✗ MISSING"
    print(f"  {status}  {series_id}  ({description})  →  {n_dcm} DICOM files")

print()
print(f"Total DICOM files to copy: {total_files}")

## Step 2 — Copy DICOM files to working directory

Copies all slices from all three series while preserving the expected directory structure:
`train_images/<study_id>/<series_id>/<instance_number>.dcm`

This is the same layout that `rsna_lumbar_spine_medgemma.ipynb` expects under `RSNA_DATA_ROOT`.

In [ ]:
import shutil

OUTPUT_ROOT    = Path("/kaggle/working/rsna_study_100206310")
OUTPUT_IMAGES  = OUTPUT_ROOT / "train_images"
OUTPUT_META    = OUTPUT_ROOT / "metadata"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_IMAGES.mkdir(parents=True, exist_ok=True)
OUTPUT_META.mkdir(parents=True, exist_ok=True)

for series_id, description in SERIES.items():
    src = study_path / series_id
    dst = OUTPUT_IMAGES / STUDY_ID / series_id

    if not src.exists():
        print(f"SKIP — source not found: {src}")
        continue

    dst.mkdir(parents=True, exist_ok=True)
    dcm_files = sorted(src.glob("*.dcm"))

    for dcm_file in dcm_files:
        shutil.copy2(dcm_file, dst / dcm_file.name)

    print(f"Copied {len(dcm_files):>3} files  →  {dst.relative_to(OUTPUT_ROOT)}  ({description})")

print()
total_copied = sum(1 for _ in (OUTPUT_IMAGES / STUDY_ID).rglob("*.dcm"))
print(f"Total DICOM files in output: {total_copied}")

## Step 3 — Copy and filter the metadata CSVs

Extracts only the rows relevant to study `100206310` from the three competition CSV files.  
These are needed by the demo notebook for:
- `train_series_descriptions.csv` — series-to-type mapping (used to load correct series)
- `train_label_coordinates.csv` — pixel coordinates for annotation overlay
- `train.csv` — ground-truth severity labels for comparison

In [ ]:
import pandas as pd

CSV_FILES = [
    ("train_series_descriptions.csv", "study_id"),
    ("train_label_coordinates.csv",   "study_id"),
    ("train.csv",                     "study_id"),
]

for csv_name, id_col in CSV_FILES:
    src_path = COMPETITION_ROOT / csv_name
    dst_path = OUTPUT_META / csv_name

    if not src_path.exists():
        print(f"SKIP — not found: {src_path}")
        continue

    df_full     = pd.read_csv(src_path)
    df_filtered = df_full[df_full[id_col].astype(str) == STUDY_ID].copy()
    df_filtered.to_csv(dst_path, index=False)

    print(f"{csv_name}")
    print(f"  Full dataset : {len(df_full):,} rows")
    print(f"  Study subset : {len(df_filtered)} rows  →  saved to {dst_path.name}")
    print()

## Step 4 — Verify output structure before zipping

In [ ]:
print(f"Output tree under: {OUTPUT_ROOT}")
print()

for root, dirs, files in os.walk(OUTPUT_ROOT):
    depth = root.replace(str(OUTPUT_ROOT), "").count(os.sep)
    indent = "    " * depth
    folder = os.path.basename(root)
    n_files = len(files)
    if depth <= 4:  # limit tree depth display
        print(f"{indent}{folder}/  ({n_files} files)")

print()

# Summarise sizes
total_bytes = sum(f.stat().st_size for f in OUTPUT_ROOT.rglob("*") if f.is_file())
print(f"Total uncompressed size: {total_bytes / 1024**2:.1f} MB")

# Show the filtered CSV contents
print()
print("=" * 55)
print("  train_series_descriptions.csv (study 100206310)")
print("=" * 55)
display(pd.read_csv(OUTPUT_META / "train_series_descriptions.csv"))

print()
print("=" * 55)
print("  train.csv row (study 100206310)")
print("=" * 55)
df_train_row = pd.read_csv(OUTPUT_META / "train.csv")
# Transpose for readability — each condition-level is a row
display(df_train_row.T.rename(columns={df_train_row.index[0]: "value"}))

print()
print("=" * 55)
print("  train_label_coordinates.csv (study 100206310)")
print("=" * 55)
df_coords = pd.read_csv(OUTPUT_META / "train_label_coordinates.csv")
print(f"  {len(df_coords)} annotation coordinates")
display(df_coords.head(10))

## Step 5 — Create the ZIP archive

The ZIP is written to `/kaggle/working/` — it will appear in the **Output** tab of this notebook.

In [ ]:
import zipfile
import time

ZIP_PATH = Path("/kaggle/working/rsna_study_100206310.zip")

print(f"Creating ZIP: {ZIP_PATH}")
t0 = time.time()

all_files = sorted(OUTPUT_ROOT.rglob("*"))
file_list = [f for f in all_files if f.is_file()]

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for file_path in file_list:
        # Store with path relative to OUTPUT_ROOT so the zip extracts cleanly
        arcname = file_path.relative_to(OUTPUT_ROOT)
        zf.write(file_path, arcname)

elapsed = time.time() - t0
zip_mb  = ZIP_PATH.stat().st_size / 1024**2

print(f"Done in {elapsed:.1f}s")
print(f"Files packed   : {len(file_list)}")
print(f"ZIP size       : {zip_mb:.1f} MB")
print(f"Output path    : {ZIP_PATH}")
print()
print("Go to the Output tab of this Kaggle notebook and download rsna_study_100206310.zip")

## Step 6 — Local extraction instructions

After downloading the ZIP to your local machine, extract it into the demo notebook's data folder:

```bash
# Create the target directory
mkdir -p ./Training/AI_In_HealthCare/ImageDiagnosis/data/rsna_study_100206310

# Extract
unzip rsna_study_100206310.zip \
  -d ./Training/AI_In_HealthCare/ImageDiagnosis/data/rsna_study_100206310
```

After extraction the layout will be:
```
data/rsna_study_100206310/
├── metadata/
│   ├── train.csv
│   ├── train_label_coordinates.csv
│   └── train_series_descriptions.csv
└── train_images/
    └── 100206310/
        ├── 1012284084/   ← Axial T2          (*.dcm)
        ├── 1792451510/   ← Sagittal T2/STIR  (*.dcm)
        └── 2092806862/   ← Sagittal T1       (*.dcm)
```

Then in `rsna_lumbar_spine_medgemma.ipynb`, update the configuration cell:
```python
RSNA_DATA_ROOT = Path("./data/rsna_study_100206310")
```

That single change is enough — the notebook auto-discovers the series from `train_series_descriptions.csv` and builds all paths from `RSNA_DATA_ROOT`.

---

## Optional — Slice-limited variant (smaller download)

If you only want the first `N` slices per series (e.g. for an even lighter download), run the cell below **instead of** Step 2.  
It respects the same output structure so all downstream code works unchanged.

In [ ]:
# ── Slice-limited copy (optional — run instead of cell in Step 2) ──────────
# Adjust MAX_SLICES_PER_SERIES to taste:
#   - 10 slices ≈ 5–15 MB per series (great for quick API tests)
#   - 20 slices ≈ 10–30 MB per series (good balance)
#   - None      = copy all slices (default — used in Step 2)

MAX_SLICES_PER_SERIES = 15

OUTPUT_ROOT_LTD   = Path("/kaggle/working/rsna_study_100206310_limited")
OUTPUT_IMAGES_LTD = OUTPUT_ROOT_LTD / "train_images"

OUTPUT_ROOT_LTD.mkdir(parents=True, exist_ok=True)
OUTPUT_IMAGES_LTD.mkdir(parents=True, exist_ok=True)

for series_id, description in SERIES.items():
    src      = study_path / series_id
    dst      = OUTPUT_IMAGES_LTD / STUDY_ID / series_id
    if not src.exists():
        print(f"SKIP — {src}")
        continue

    dst.mkdir(parents=True, exist_ok=True)
    all_dcm  = sorted(src.glob("*.dcm"))
    # Evenly sample MAX_SLICES_PER_SERIES from the full slice range
    import numpy as np
    if MAX_SLICES_PER_SERIES and len(all_dcm) > MAX_SLICES_PER_SERIES:
        indices = np.linspace(0, len(all_dcm) - 1, MAX_SLICES_PER_SERIES, dtype=int)
        selected = [all_dcm[i] for i in indices]
    else:
        selected = all_dcm

    for dcm_file in selected:
        shutil.copy2(dcm_file, dst / dcm_file.name)

    print(f"Copied {len(selected):>3}/{len(all_dcm)} slices  →  {dst.relative_to(OUTPUT_ROOT_LTD)}  ({description})")

# Copy metadata too
OUTPUT_META_LTD = OUTPUT_ROOT_LTD / "metadata"
OUTPUT_META_LTD.mkdir(exist_ok=True)
for csv_name, _ in CSV_FILES:
    src_csv = OUTPUT_META / csv_name  # already filtered in Step 3
    if src_csv.exists():
        shutil.copy2(src_csv, OUTPUT_META_LTD / csv_name)

# Zip the limited version
ZIP_LTD = Path("/kaggle/working/rsna_study_100206310_limited.zip")
limited_files = [f for f in OUTPUT_ROOT_LTD.rglob("*") if f.is_file()]
with zipfile.ZipFile(ZIP_LTD, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for fp in limited_files:
        zf.write(fp, fp.relative_to(OUTPUT_ROOT_LTD))

total_ltd = sum(f.stat().st_size for f in OUTPUT_ROOT_LTD.rglob("*") if f.is_file())
print()
print(f"Limited ZIP: {ZIP_LTD.name}  ({ZIP_LTD.stat().st_size / 1024**2:.1f} MB compressed, "
      f"{total_ltd / 1024**2:.1f} MB uncompressed, {len(limited_files)} files)")